# MTG Commander — Mistral-Nemo-12B QLoRA fine-tune

Trains the **v2** model (`mistral-commander-nemo`) from `mistralai/Mistral-Nemo-Instruct-2407`.

**Auto-detects your Colab GPU** and picks the right config:

| GPU | VRAM | Method | Train ctx | LoRA r | Per-device batch | Effective batch |
|---|---|---|---|---|---|---|
| T4 | 16 GB | QLoRA NF4 | 4096 | 32 | 1 | 16 |
| L4 | 24 GB | QLoRA NF4 | 8192 | 32 | 1 | 16 |
| A100-40 | 40 GB | bf16 LoRA | 8192 | 64 | 2 | 16 |
| A100-80 | 80 GB | bf16 LoRA | 16384 | 128 | 4 | 16 |

**Drive checkpointing** every ~250 steps so Colab disconnects don't kill progress.

## Before running
1. Accept the gated model license: <https://huggingface.co/mistralai/Mistral-Nemo-Instruct-2407>
2. Create a HF token (Read scope): <https://huggingface.co/settings/tokens>
3. Build dataset v3 locally (see `DATASET_V3.md`) and upload `chat_train.jsonl` + `chat_eval.jsonl` to your Drive at `MyDrive/mtg-training/dataset_v3/`.
4. **Runtime → Change runtime type → GPU** (Pro: prefer L4 or A100; Free: T4 only).

## 1. Mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pinned versions known to work with Mistral-Nemo + QLoRA as of 2026-04.
# Bump cautiously; trl/peft API churn breaks older notebooks regularly.
%pip install -q --upgrade pip
%pip install -q \
  "transformers>=4.46.0" \
  "trl>=0.12.0" \
  "peft>=0.13.0" \
  "accelerate>=1.0.0" \
  "datasets>=3.0.0" \
  "bitsandbytes>=0.44.0" \
  "sentencepiece>=0.2.0" \
  "huggingface_hub[cli]>=0.25.0" \
  "safetensors>=0.4.5"

## 2. Hugging Face login

Paste your `hf_...` token when prompted. Required to download the gated Mistral-Nemo base.

In [ ]:
from huggingface_hub import login
login()  # opens an inline prompt

## 3. Detect GPU and pick training profile

In [ ]:
import torch, subprocess, json

assert torch.cuda.is_available(), 'No GPU detected — switch runtime to GPU.'

name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {name}  ({vram_gb:.1f} GB)')

# Profiles keyed by VRAM, since the GPU name is unreliable across Colab regions.
if vram_gb >= 75:
    PROFILE = dict(label='A100-80', use_qlora=False, max_seq_len=16384, lora_r=128, lora_alpha=256, per_device_bs=4, grad_accum=4)
elif vram_gb >= 38:
    PROFILE = dict(label='A100-40', use_qlora=False, max_seq_len=8192,  lora_r=64,  lora_alpha=128, per_device_bs=2, grad_accum=8)
elif vram_gb >= 22:
    PROFILE = dict(label='L4',      use_qlora=True,  max_seq_len=8192,  lora_r=32,  lora_alpha=64,  per_device_bs=1, grad_accum=16)
else:
    PROFILE = dict(label='T4',      use_qlora=True,  max_seq_len=4096,  lora_r=32,  lora_alpha=64,  per_device_bs=1, grad_accum=16)

print(json.dumps(PROFILE, indent=2))

## 4. Paths + config

Edit `RUN_NAME` if you want side-by-side experiments — checkpoints go under `MyDrive/mtg-training/runs/<RUN_NAME>/`.

In [ ]:
from pathlib import Path

BASE_MODEL    = 'mistralai/Mistral-Nemo-Instruct-2407'
RUN_NAME      = 'nemo-v2-sft'
EPOCHS        = 2
LR            = 2e-4

DRIVE_ROOT    = Path('/content/drive/MyDrive/mtg-training')
DATASET_DIR   = DRIVE_ROOT / 'dataset_v3'
TRAIN_FILE    = DATASET_DIR / 'chat_train.jsonl'
EVAL_FILE     = DATASET_DIR / 'chat_eval.jsonl'
RUN_DIR       = DRIVE_ROOT / 'runs' / RUN_NAME
CKPT_DIR      = RUN_DIR / 'checkpoints'
ADAPTER_DIR   = RUN_DIR / 'adapter'
MERGED_DIR    = RUN_DIR / 'merged'
GGUF_DIR      = RUN_DIR / 'gguf'

for d in (CKPT_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert TRAIN_FILE.exists(), f'Missing {TRAIN_FILE} — upload your dataset_v3 to Drive first.'
print('Train:', TRAIN_FILE, TRAIN_FILE.stat().st_size, 'bytes')
print('Eval :', EVAL_FILE if EVAL_FILE.exists() else '(none)')
print('Run  :', RUN_DIR)

## 5. Load tokenizer and base model

First call downloads ~24 GB to the Colab VM (3–5 min). Subsequent runs in the same session are cached.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

if PROFILE['use_qlora']:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_cfg,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        attn_implementation='sdpa',
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        attn_implementation='sdpa',
    )

model.config.use_cache = False  # required for gradient checkpointing
print('Loaded base. dtype:', next(model.parameters()).dtype)

## 6. Load dataset and apply Nemo chat template

In [ ]:
from datasets import load_dataset

data_files = {'train': str(TRAIN_FILE)}
if EVAL_FILE.exists():
    data_files['eval'] = str(EVAL_FILE)

raw = load_dataset('json', data_files=data_files)

def format_row(row):
    text = tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

train_ds = raw['train'].map(format_row, remove_columns=raw['train'].column_names)
eval_ds  = raw['eval' ].map(format_row, remove_columns=raw['eval' ].column_names) if 'eval' in raw else None

print('Train rows:', len(train_ds))
print('Eval  rows:', len(eval_ds) if eval_ds else 0)
print('--- example ---')
print(train_ds[0]['text'][:600])

## 7. Configure LoRA and trainer

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

if PROFILE['use_qlora']:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()

peft_cfg = LoraConfig(
    r=PROFILE['lora_r'],
    lora_alpha=PROFILE['lora_alpha'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

sft_cfg = SFTConfig(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    per_device_train_batch_size=PROFILE['per_device_bs'],
    per_device_eval_batch_size=PROFILE['per_device_bs'],
    gradient_accumulation_steps=PROFILE['grad_accum'],
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    optim='paged_adamw_8bit' if PROFILE['use_qlora'] else 'adamw_torch',
    max_seq_length=PROFILE['max_seq_len'],
    packing=False,
    dataset_text_field='text',
    logging_steps=10,
    save_strategy='steps',
    save_steps=250,
    save_total_limit=3,
    eval_strategy='steps' if eval_ds else 'no',
    eval_steps=250 if eval_ds else None,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_cfg,
    args=sft_cfg,
)
trainer.model.print_trainable_parameters()

## 8. Train (resumes from latest Drive checkpoint if present)

**This cell can take many hours.** Drive checkpoints save every 250 steps, so if Colab kicks you off you can:
1. Reconnect to a runtime,
2. Re-run cells 1–7,
3. Re-run this cell — `resume_from_checkpoint=True` picks up where it left off.

In [ ]:
from pathlib import Path

has_ckpt = any(p.is_dir() and p.name.startswith('checkpoint-') for p in CKPT_DIR.iterdir()) if CKPT_DIR.exists() else False
trainer.train(resume_from_checkpoint=True if has_ckpt else None)

trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('Saved final adapter ->', ADAPTER_DIR)

## 9. Quick smoke test

Sanity check the freshly-trained adapter on a deckbuilding prompt before merging.

In [ ]:
from transformers import pipeline

messages = [
    {'role': 'system', 'content': 'You are an expert Magic: The Gathering Commander deck builder.'},
    {'role': 'user', 'content': 'Commander: Atraxa, Praetors\' Voice\nColor identity: W, U, B, G\nStrategy: +1/+1 counters with proliferate.\nRespond with a JSON object containing description and a 99-card deck.'},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

import torch
with torch.no_grad():
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## 10. Merge LoRA into base weights

Required step before GGUF conversion. Produces ~24 GB of merged bf16 weights on the Colab VM.

**Tip:** if you're tight on disk, skip the Drive copy and run the conversion + quantize in the next cell, then upload only the small `.gguf` to Drive.

In [ ]:
# Free VRAM before reloading at full precision
import gc, torch
del trainer, model
gc.collect(); torch.cuda.empty_cache()

from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='cpu',  # merge on CPU to avoid OOM, slower but reliable
)
merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
merged = merged.merge_and_unload()

MERGED_LOCAL = Path('/content/merged')
MERGED_LOCAL.mkdir(exist_ok=True)
merged.save_pretrained(str(MERGED_LOCAL), safe_serialization=True, max_shard_size='4GB')
tokenizer.save_pretrained(str(MERGED_LOCAL))
print('Merged weights at', MERGED_LOCAL)

## 11. Convert merged model to GGUF and quantize to Q4_K_M

Final artifact `mtg-commander-nemo-q4.gguf` (~7.5 GB) — drops straight into Ollama or llama.cpp.

In [ ]:
%cd /content
![ -d llama.cpp ] || git clone --depth=1 https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!pip install -q -r requirements.txt
# Build the quantize binary (CMake flow, llama.cpp dropped Make)
!cmake -B build -DGGML_CUDA=OFF >/dev/null && cmake --build build --target llama-quantize -j 2

In [ ]:
F16_GGUF = Path('/content/mtg-commander-nemo-f16.gguf')
Q4_GGUF  = Path('/content/mtg-commander-nemo-q4_k_m.gguf')

!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/merged \
    --outfile {F16_GGUF} \
    --outtype f16

!/content/llama.cpp/build/bin/llama-quantize {F16_GGUF} {Q4_GGUF} Q4_K_M

import shutil
shutil.copy(Q4_GGUF, GGUF_DIR / Q4_GGUF.name)
print('Copied Q4_K_M GGUF to Drive:', GGUF_DIR / Q4_GGUF.name)

## 12. (Optional) Upload to Hugging Face

Pushes the GGUF + the LoRA adapter to a new HF repo. Set `HF_REPO` to your username/repo.

In [ ]:
HF_REPO = 'SaltyNumba1/mtg-commander-nemo'  # change if you want a different repo

from huggingface_hub import HfApi, create_repo
api = HfApi()
create_repo(HF_REPO, repo_type='model', exist_ok=True)

api.upload_file(path_or_fileobj=str(Q4_GGUF), path_in_repo=Q4_GGUF.name, repo_id=HF_REPO)
api.upload_folder(folder_path=str(ADAPTER_DIR), path_in_repo='lora_adapter', repo_id=HF_REPO)
print('Uploaded to', f'https://huggingface.co/{HF_REPO}')

## 13. Local install (after download)

On your machine:

```powershell
# 1. Download the .gguf from Drive or HF, place it next to the Modelfile.
# 2. Edit Modelfile: change `FROM ./mistral-commander-q4.gguf`
#                       to `FROM ./mtg-commander-nemo-q4_k_m.gguf`
# 3. Build the Ollama model with a new name:
ollama create mtg-commander-nemo -f Modelfile
# 4. Point the app at it (PowerShell, current session):
$env:OLLAMA_MODEL = 'mtg-commander-nemo'
# 5. Or persist by adding OLLAMA_MODEL=mtg-commander-nemo to a .env
#    file next to MTG Collection.exe.
```

v1 (`mtg-commander`) stays installed as a fallback — you can flip between them with the env var.